# 🧠 Methodology Classifier – v2.2 (SciBERT + XGBoost)
This experiment tests whether domain-specific contextual embeddings from SciBERT combined with a non-linear classifier (XGBoost) can improve Methodology classification accuracy beyond 71%, targeting 90–95% range.

## Imports

In [ ]:
# Basic imports
import pandas as pd
import numpy as np

# For embeddings
from sentence_transformers import SentenceTransformer

# For preprocessing and classification
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier

# For saving models
import joblib

## Load dataset and preprocess text

In [ ]:
# Load dataset
df = pd.read_csv("Data/NLP_Dataset_Title_Abstract_Discipline_Subfield_Methodology.csv")

# Combine Title + Abstract
df["text"] = df["Title"].fillna('') + " " + df["Abstract"].fillna('')
df["text"] = df["text"].str.strip()

# Drop rows with missing Methodology
df = df.dropna(subset=["Methodology"])

# Extract inputs and labels
texts = df["text"].tolist()
labels = df["Methodology"].tolist()

In [ ]:
# Show first 3 rows
print("✅ Sample rows:")
display(df[["Title", "Abstract", "text", "Methodology"]].head(3))

# Show number of samples and class distribution
print("\n📊 Dataset size:", len(df))
print("\n🔢 Methodology class distribution:")
print(df["Methodology"].value_counts())

## Generate SciBERT embeddings

In [ ]:
model = SentenceTransformer('allenai/scibert_scivocab_uncased')
X = model.encode(texts, show_progress_bar=True)

## Encode Labels 

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)

# Check label encoding
print("🔠 Label classes:", label_encoder.classes_)
print("🔢 Encoded values:", np.unique(y))

## Train/test split + scaling

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 80/20 split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Scale the dense vectors
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Train XGBoost Classifier

In [ ]:
from xgboost import XGBClassifier

# Initialize and train XGBoost
clf = XGBClassifier(eval_metric='mlogloss', random_state=42)
clf.fit(X_train_scaled, y_train)

## Evaluate Model Performance

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Predict on test set
y_pred = clf.predict(X_test_scaled)

# Evaluate
print("✅ Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\n📄 Classification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

## Save model and other artefacts

In [ ]:
joblib.dump(clf, "Artefacts/methodology_scibert_xgb_v2.2_model.pkl")
joblib.dump(scaler, "Artefacts/methodology_scibert_xgb_v2.2_scaler.pkl")
joblib.dump(label_encoder, "Artefacts/methodology_scibert_xgb_v2.2_label_encoder.pkl")

## 🔍 Results Summary – v2.2 (SciBERT + XGBoost, 80/20 Split)

- **Accuracy**: 0.667
- **Macro F1**: 0.47
- **Best Class**: Quantitative (F1 = 0.70)
- **Qualitative**: F1 = 0.70
- **Mixed**: Not predicted at all (F1 = 0.00)

### 🔎 Observations:
- Model performed reasonably well on QLT and QNT classes.
- Mixed Methods (M) was completely missed — likely due to only 2 training examples.
- SciBERT + XGBoost provided a strong semantic baseline, slightly underperforming the TF-IDF + SVM setup (v2.0: 0.71).
- Scaling was critical to stabilizing classifier training.
- This run establishes a reproducible BERT baseline for future improvements via SMOTE, hyperparameter tuning, or cross-validation.

✅ Model, scaler, and label encoder saved as versioned artefacts in `Artefacts/`

## Apply SMOTE to Training Data and Retrain XGBoost on SMOTE Data

In [ ]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE with k=1 (good for very small classes like Mixed)
smote = SMOTE(random_state=42, k_neighbors=1)

X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

# Check new class distribution
import collections
print("🔁 Resampled class distribution:", collections.Counter(y_train_resampled))

# Retrain model on resampled data
clf_smote = XGBClassifier(eval_metric='mlogloss', random_state=42)
clf_smote.fit(X_train_resampled, y_train_resampled)

## Evaluate Model Performance and Classification Report

In [ ]:
y_pred_smote = clf_smote.predict(X_test_scaled)

print("✅ Accuracy (after SMOTE):", round(accuracy_score(y_test, y_pred_smote), 4))
print("\n📄 Classification Report (after SMOTE):")
print(classification_report(y_test, y_pred_smote, target_names=label_encoder.classes_))

## Save model

In [ ]:
joblib.dump(clf_smote, "Artefacts/methodology_scibert_xgb_v2.2.1_smote_model.pkl")

## 🔁 v2.2.1 (SciBERT + XGBoost + SMOTE)

- Accuracy: 76.19%
- Macro F1: 0.54
- Weighted F1: 0.74
- Mixed Methods: still 0.00 (likely due to 2 test samples)

##  SMOTE + XGBoost with Manual 5-Fold Cross-Validation

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from imblearn.over_sampling import SMOTE

# Prepare CV loop
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accuracy_scores = []
macro_f1_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    # Split and scale
    X_train_fold, X_val_fold = X[train_idx], X[val_idx]
    y_train_fold, y_val_fold = y[train_idx], y[val_idx]
    
    # Scale embeddings
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_fold)
    X_val_scaled = scaler.transform(X_val_fold)
    
    # Apply SMOTE
    smote = SMOTE(random_state=42, k_neighbors=1)
    X_resampled, y_resampled = smote.fit_resample(X_train_scaled, y_train_fold)
    
    # Train XGBoost
    clf = XGBClassifier(eval_metric='mlogloss', random_state=42)
    clf.fit(X_resampled, y_resampled)
    
    # Predict
    y_pred = clf.predict(X_val_scaled)
    
    # Score
    acc = accuracy_score(y_val_fold, y_pred)
    f1 = f1_score(y_val_fold, y_pred, average='macro')
    
    accuracy_scores.append(acc)
    macro_f1_scores.append(f1)
    
    print(f"✅ Fold {fold}: Accuracy = {round(acc, 4)}, Macro F1 = {round(f1, 4)}")

# Summary
print("\n📊 Final 5-Fold CV Results:")
print("Mean Accuracy:", round(np.mean(accuracy_scores), 4))
print("Std Accuracy:", round(np.std(accuracy_scores), 4))
print("Mean Macro F1:", round(np.mean(macro_f1_scores), 4))
print("Std Macro F1:", round(np.std(macro_f1_scores), 4))

## 🔁 v2.2.1 Cross-Validation Results – SciBERT + XGBoost + SMOTE

- 5-fold Stratified CV performed on full dataset
- SMOTE applied within each fold to balance all 3 classes
- Classifier: XGBoost on scaled SciBERT embeddings (768-dim)

### 📊 Cross-Validation Summary:
- **Mean Accuracy**: 0.6571
- **Std Dev (Accuracy)**: 0.1017
- **Mean Macro F1**: 0.5373
- **Std Dev (Macro F1)**: 0.1028

### 🧠 Observations:
- Performance is consistent across folds despite M class difficulty
- Best macro F1 across all BERT-based versions so far
- Establishes a robust semantic + balanced baseline before hyperparameter tuning or fine-tuning